# 03 — Structural curation audit

**Workflow version:** 0.5.2

Audit the released original and curated SBML models directly. This notebook is independent of phenotype accuracy and asks whether reaction bounds, GPRs, added reactions, and selected stoichiometric edits changed as expected. It also contains focused diagnostics for phenotype isolation failures discovered during notebook 02 validation.


## Notebook navigation convention

Every code cell starts with three navigation comments: **Code Cell number**, **Requires**, and **Output/Provides**. Use these labels instead of Jupyter execution counters such as `[3]` or `[12]`, because execution counters change after kernel restarts.


In [ ]:
# Code Cell 1 — Setup paths and load released models
# Requires: nothing
# Provides: ROOT, DATA_DIR, RESULTS_DIR, original, curated, pairwise

from pathlib import Path
import numpy as np
import pandas as pd
from cobra.io import read_sbml_model

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = ROOT / "data" / "raw"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

original = read_sbml_model(str(DATA_DIR / "yeast9.0.xml"))
curated = read_sbml_model(str(DATA_DIR / "Yeast9_curated.xml"))

PAIRWISE_PATH = RESULTS_DIR / "02_pairwise_comparison.csv"
if not PAIRWISE_PATH.exists():
    raise FileNotFoundError("Run notebook 02 first so 02_pairwise_comparison.csv is available.")

pairwise = pd.read_csv(PAIRWISE_PATH)

In [ ]:
# Code Cell 2 — Build the structural curation audit
# Requires: Code Cell 1
# Output: results/03_structural_curation_audit.csv

CURATION_REACTIONS = [
    "r_0217", "r_0172",
    "r_2488", "r_2489", "r_2490", "r_2491",
    "r_2492", "r_2493", "r_2494", "r_2495",
    "r_0312", "r_4702", "r_4703", "r_0559",
    "r_4048", "r_4598", "r_1026", "r_0080", "r_0815",
    "r_2067", "r_2070", "r_2071", "r_0250",
    "r_temp1", "r_temp2", "r_temp3",
]

def snapshot(model, reaction_id):
    if reaction_id not in model.reactions:
        return {
            "reaction": reaction_id, "present": False,
            "lower_bound": np.nan, "upper_bound": np.nan,
            "gpr": "", "reaction_name": "", "equation": "",
        }
    reaction = model.reactions.get_by_id(reaction_id)
    return {
        "reaction": reaction_id, "present": True,
        "lower_bound": reaction.lower_bound,
        "upper_bound": reaction.upper_bound,
        "gpr": reaction.gene_reaction_rule,
        "reaction_name": reaction.name,
        "equation": reaction.reaction,
    }

rows = []
for reaction_id in CURATION_REACTIONS:
    rows.append({"model": "Yeast9", **snapshot(original, reaction_id)})
    rows.append({"model": "Yeast9_curated", **snapshot(curated, reaction_id)})

structural_audit = pd.DataFrame(rows)
display(structural_audit)
structural_audit.to_csv(RESULTS_DIR / "03_structural_curation_audit.csv", index=False)

## Regression-focused structural audit

Notebook 02 produced three curated-model regressions. This section records their phenotype values and inspects the gene-reaction structure around the affected arginine/uracil and thiamine cases. The tables are evidence for diagnosis; they do not by themselves establish causality.


In [ ]:
# Code Cell 3 — Extract provisional regression pairs from Notebook 02
# Requires: Code Cell 1 and results/02_pairwise_comparison.csv
# Output: results/03_regression_pairs.csv

regression_columns = [
    "pair_id",
    "excel_row_original",
    "gene_field_original",
    "chemical_original",
    "classification_original",
    "classification_curated",
    "ko_growth_original",
    "ko_growth_curated",
    "rescue_growth_original",
    "rescue_growth_curated",
    "mapped_background_original",
    "mapped_background_curated",
    "mapped_rescue_original",
    "mapped_rescue_curated",
]

regressions = pairwise.loc[pairwise["change"] == "regression", regression_columns].copy()

if len(regressions) != 3:
    raise AssertionError(f"Expected 3 regressions from notebook 02, found {len(regressions)}.")

display(regressions)
regressions.to_csv(RESULTS_DIR / "03_regression_pairs.csv", index=False)


In [ ]:
# Code Cell 4 — Audit gene-reaction structure for regression-related genes
# Requires: Code Cell 1
# Output: results/03_regression_gene_reaction_audit.csv

FOCUS_GENES = ["YOR303W", "YJR109C", "YJL130C", "YPL214C", "YGR144W"]

def gene_reaction_snapshot(model, model_label, gene_id):
    if gene_id not in model.genes:
        return [{
            "model": model_label,
            "gene": gene_id,
            "reaction": "",
            "present": False,
            "lower_bound": np.nan,
            "upper_bound": np.nan,
            "gpr": "",
            "equation": "",
        }]

    gene = model.genes.get_by_id(gene_id)
    reactions = sorted(gene.reactions, key=lambda reaction: reaction.id)

    if not reactions:
        return [{
            "model": model_label,
            "gene": gene_id,
            "reaction": "",
            "present": True,
            "lower_bound": np.nan,
            "upper_bound": np.nan,
            "gpr": "",
            "equation": "",
        }]

    return [
        {
            "model": model_label,
            "gene": gene_id,
            "reaction": reaction.id,
            "present": True,
            "lower_bound": reaction.lower_bound,
            "upper_bound": reaction.upper_bound,
            "gpr": reaction.gene_reaction_rule,
            "equation": reaction.reaction,
        }
        for reaction in reactions
    ]

gene_rows = []
for gene_id in FOCUS_GENES:
    gene_rows.extend(gene_reaction_snapshot(original, "Yeast9", gene_id))
    gene_rows.extend(gene_reaction_snapshot(curated, "Yeast9_curated", gene_id))

gene_reaction_audit = pd.DataFrame(gene_rows)
display(gene_reaction_audit)
gene_reaction_audit.to_csv(RESULTS_DIR / "03_regression_gene_reaction_audit.csv", index=False)


In [ ]:
# Code Cell 5 — Audit regression-focused reactions
# Requires: Code Cell 2
# Output: results/03_regression_reaction_audit.csv

REGRESSION_FOCUS_REACTIONS = [
    "r_0250",
    "r_2067",
    "r_2070",
    "r_2071",
    "r_temp2",
    "r_temp3",
]

regression_reaction_audit = structural_audit.loc[
    structural_audit["reaction"].isin(REGRESSION_FOCUS_REACTIONS),
    [
        "model",
        "reaction",
        "present",
        "lower_bound",
        "upper_bound",
        "gpr",
        "reaction_name",
        "equation",
    ],
].copy()

display(regression_reaction_audit)
regression_reaction_audit.to_csv(
    RESULTS_DIR / "03_regression_reaction_audit.csv",
    index=False,
)


In [ ]:
# Code Cell 6 — Compare selected stoichiometric changes
# Requires: Code Cell 1
# Output: results/03_stoichiometric_changes.csv

def stoichiometry_by_id(reaction):
    return {metabolite.id: float(coefficient) for metabolite, coefficient in reaction.metabolites.items()}

def stoichiometric_diff(original_model, curated_model, reaction_id):
    if reaction_id not in original_model.reactions or reaction_id not in curated_model.reactions:
        return pd.DataFrame()
    a = stoichiometry_by_id(original_model.reactions.get_by_id(reaction_id))
    b = stoichiometry_by_id(curated_model.reactions.get_by_id(reaction_id))
    metabolite_ids = sorted(set(a) | set(b))
    rows = []
    for metabolite_id in metabolite_ids:
        old = a.get(metabolite_id, 0.0)
        new = b.get(metabolite_id, 0.0)
        if not np.isclose(old, new, atol=1e-12):
            rows.append({
                "reaction": reaction_id,
                "metabolite": metabolite_id,
                "original_coefficient": old,
                "curated_coefficient": new,
                "delta": new - old,
            })
    return pd.DataFrame(rows)

stoich_changes = pd.concat(
    [stoichiometric_diff(original, curated, rid) for rid in ["r_4048", "r_4598"]],
    ignore_index=True,
)

display(stoich_changes)
stoich_changes.to_csv(RESULTS_DIR / "03_stoichiometric_changes.csv", index=False)

In [ ]:
# Code Cell 7 — Display the focused structural comparison
# Requires: Code Cell 2
# Output: display only

focus = structural_audit[structural_audit["reaction"].isin(["r_0217", "r_0172", "r_4702", "r_4703", "r_0080", "r_0250", "r_temp1", "r_temp2", "r_temp3"])]
display(
    focus.pivot(
        index="reaction",
        columns="model",
        values=["present", "lower_bound", "upper_bound", "gpr"],
    )
)

## Fresh-process isolation diagnostic

The reference benchmark cannot currently rely on `Model.copy()`, sequential model contexts, or JSON reconstruction. This diagnostic runs each selected phenotype in a separate Python process. Each worker process loads the released SBML exactly once, performs one phenotype simulation, returns a JSON record, and exits.

The purpose of this section is to validate process-level isolation on difficult cases before changing notebook 02 or rerunning the full 147-pair benchmark.


In [ ]:
# Code Cell 8 — Run the fresh-process isolation diagnostic
# Requires: Code Cell 1 only
# Output: results/03_process_isolation_diagnostic.csv

import json
import subprocess
import sys

PROCESS_WORKER = ROOT / "scripts" / "fresh_pair_worker.py"
CURATED_PATH = DATA_DIR / "Yeast9_curated.xml"

if not PROCESS_WORKER.exists():
    raise FileNotFoundError(f"Missing process worker: {PROCESS_WORKER}")

original_for_threshold = read_sbml_model(str(DATA_DIR / "yeast9.0.xml"))
original_for_threshold.solver = "glpk"
reference_threshold = 0.01 * float(
    original_for_threshold.slim_optimize(error_value=np.nan)
)

PROCESS_CASES = [
    {
        "pair_id": "diagnostic:YGR204W",
        "gene": ["YGR204W"],
        "background": [],
        "rescue": ["r_1893", "r_1639"],
    },
    {
        "pair_id": "diagnostic:YGR144W",
        "gene": ["YGR144W"],
        "background": [],
        "rescue": ["r_2067"],
    },
    {
        "pair_id": "diagnostic:YPL214C",
        "gene": ["YPL214C"],
        "background": [],
        "rescue": ["r_2067"],
    },
    {
        "pair_id": "diagnostic:YPL028W",
        "gene": ["YPL028W"],
        "background": [],
        "rescue": ["r_1757"],
    },
    {
        "pair_id": "diagnostic:YOR303W",
        "gene": ["YOR303W"],
        "background": ["r_2090"],
        "rescue": ["r_1879"],
    },
    {
        "pair_id": "diagnostic:YJR109C",
        "gene": ["YJR109C"],
        "background": ["r_2090"],
        "rescue": ["r_1879"],
    },
]


def run_fresh_process_case(case):
    command = [
        sys.executable,
        str(PROCESS_WORKER),
        "--model-path",
        str(CURATED_PATH),
        "--model-label",
        "Yeast9_curated",
        "--solver",
        "glpk",
        "--threshold",
        repr(reference_threshold),
        "--uptake-lower-bound",
        "-1000",
        "--aliases-json",
        json.dumps({"a_0001": "r_temp1"}),
        "--pair-id",
        case["pair_id"],
    ]

    for gene_id in case["gene"]:
        command.extend(["--gene", gene_id])

    for reaction_id in case["background"]:
        command.extend(["--background", reaction_id])

    for reaction_id in case["rescue"]:
        command.extend(["--rescue", reaction_id])

    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=True,
        timeout=180,
    )

    lines = [line.strip() for line in completed.stdout.splitlines() if line.strip()]
    if not lines:
        raise RuntimeError(
            f"Worker returned no JSON output for {case['pair_id']}. "
            f"stderr={completed.stderr}"
        )

    return json.loads(lines[-1])


process_results = []

for index, case in enumerate(PROCESS_CASES, start=1):
    print(
        f"START {index}/{len(PROCESS_CASES)} | "
        f"{case['pair_id']} | {case['gene'][0]}",
        flush=True,
    )

    result = run_fresh_process_case(case)
    process_results.append(result)

    print(
        f"DONE  {index}/{len(PROCESS_CASES)} | "
        f"class={result['classification']} | "
        f"KO={result['ko_status']}:{result['ko_growth']} | "
        f"rescue={result['rescue_status']}:{result['rescue_growth']} | "
        f"pid={result['pid']}",
        flush=True,
    )

process_diagnostic = pd.DataFrame(process_results)

display(
    process_diagnostic[
        [
            "pair_id",
            "pid",
            "genes",
            "associated_reactions",
            "classification",
            "ko_status",
            "ko_growth",
            "rescue_status",
            "rescue_growth",
            "mapped_background",
            "mapped_rescue",
        ]
    ]
)

process_diagnostic.to_csv(
    RESULTS_DIR / "03_process_isolation_diagnostic.csv",
    index=False,
)

if process_diagnostic["pid"].nunique() != len(process_diagnostic):
    raise AssertionError("Each diagnostic case must run in a distinct process.")


### Acceptance checkpoint

Do not promote process-level isolation to the reference benchmark until these six cases reproduce the previously observed fresh-SBML classifications and growth behavior. In particular, `YPL214C` must be rescued from zero knockout growth, `YPL028W` must be rescued by ergosterol, and `YOR303W` / `YJR109C` must remain Type I under the released curated `r_0250` GPR.


## Interpretation checkpoint

This notebook reports **released-artifact structure**. A mismatch between the paper description and the released curated SBML is an artifact-level reproducibility observation; it is not by itself evidence about author intent.